# Week 04 — Spatial EDA and Neighbourhoods

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Understanding Exploratory Data Analysis · Spatial autocorrelation, contiguity, Moran’s I

**Learning outcome.** Compare neighboring temperatures and permutation significance.

**What you will implement:** Rook spatial weights, spatial lag, Moran permutation test.  
**Mathematical model:** Moran I = (n/S₀)·ΣᵢΣⱼ wᵢⱼzᵢzⱼ/Σᵢzᵢ².

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Moran I = (n/S₀)·ΣᵢΣⱼ wᵢⱼzᵢzⱼ/Σᵢzᵢ²

**Method family:** Rook spatial weights, spatial lag, Moran permutation test.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
cells=pd.read_csv(DATA/'cells.csv');obs=pd.read_csv(DATA/'observations_clean.csv')
june=cells.merge(obs[obs.date=='2025-06-01'],on='cell_id').sort_values(['row','col'])
n=len(june);W=np.zeros((n,n))
for i in range(n):
    r,c=divmod(i,12)
    for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr,nc=r+dr,c+dc
        if 0<=nr<12 and 0<=nc<12: W[i,nr*12+nc]=1
W=W/W.sum(axis=1,keepdims=True)
y=june.lst_c.to_numpy();z=y-y.mean();lag=W@z
I=(n/W.sum())*((z@lag)/(z@z));print('Global rook Moran I',round(I,3))

### Reading the main code

**Weights.** Rows and columns are sorted before constructing the `W` matrix. A cell receives weight 1 for each rook neighbour, then weights are normalized by the row sum.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `merge` — Joins tables on a declared key; validate cardinality to avoid duplicate matches.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
rng=np.random.default_rng(2026)
draws=[]
for _ in range(999):
    zp=rng.permutation(z)
    draws.append((n/W.sum())*(zp@(W@zp))/(zp@zp))
p_plus=(1+np.count_nonzero(np.array(draws)>=I))/(1+len(draws))
print('One-sided permutation p:',p_plus)
plt.scatter(z,lag,alpha=.65);plt.axhline(0,color='gray');plt.axvline(0,color='gray')
plt.xlabel('Centered value');plt.ylabel('Rook spatial lag');plt.title('Moran scatter');plt.show()

### Reading the output code

**Statistic.** `z` centers observations, `W@z` creates their spatial lag, and `I` compares neighbour similarity to the total variance.

- `np.count_nonzero` — Counts True Boolean conditions for confusion-matrix or IoU calculations.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Permutation.** Shuffle `z` 999 times with a fixed RNG; `p_plus` counts simulated I at least as large as the observed one and uses the +1 correction.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Compare rook with queen neighbours. Which geographic hypothesis changes, and why?
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.